# Audio lab: inspect and paint a real sound image

This optional notebook starts from a **real five-second training recording**. Its detailed time-frequency image is the canvas: you can strengthen or erase local features, view the changed image, and hear the result. The first plot also shows a 64-band log-mel image like the input to a 2D audio CNN. Run the notebook from the `student_release` folder.


In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal
from IPython.display import Audio, display


## 1. Choose a training recording

Change `example_index` to explore another anonymous training clip. The notebook never uses validation or test audio.


In [ ]:
with Path("dataset/manifest.csv").open(newline="") as f:
    train_rows = [row for row in csv.DictReader(f) if row["split"] == "train"]
example_index = 0
row = train_rows[example_index]
sr, pcm = wavfile.read(Path("dataset/audio") / row["id"])
x = pcm.astype(np.float32) / 32768.0
assert sr == 16000 and x.ndim == 1 and len(x) == 5 * sr
print(f"Anonymous clip: {row['id']}  label: {row['label']}  duration: {len(x)/sr:.1f} s")
display(Audio(x, rate=sr))


## 2. See the waveform, STFT, and log-mel image

The STFT magnitude has one row per frequency bin. A mel filter bank combines nearby bins into 64 perceptual bands. Both images display energy patterns over time; neither contains the complex phase needed for an exact waveform reconstruction.


In [ ]:
n_fft = 512
hop = 128
frequency, time, spectrum = signal.stft(
    x, fs=sr, nperseg=n_fft, noverlap=n_fft-hop,
    boundary="zeros", padded=True
)
magnitude = np.abs(spectrum)
stft_db = 20 * np.log10(np.maximum(magnitude, 1e-5))

def hz_to_mel(hz):
    return 2595 * np.log10(1 + hz / 700)

def mel_to_hz(mel):
    return 700 * (10 ** (mel / 2595) - 1)

mel_points = mel_to_hz(np.linspace(hz_to_mel(0), hz_to_mel(sr/2), 66))
mel_filters = np.zeros((64, len(frequency)))
for band in range(64):
    left, center, right = mel_points[band:band+3]
    rising = (frequency-left) / (center-left)
    falling = (right-frequency) / (right-center)
    mel_filters[band] = np.maximum(0, np.minimum(rising, falling))
mel_db = 20 * np.log10(np.maximum(mel_filters @ magnitude, 1e-5))

fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
axes[0].plot(np.arange(len(x))/sr, x, lw=0.45)
axes[0].set(xlabel="Time (s)", ylabel="Amplitude", title="Waveform")
axes[1].pcolormesh(time, frequency, stft_db, shading="auto", cmap="magma", vmin=-85, vmax=0)
axes[1].set(xlabel="Time (s)", ylabel="Frequency (Hz)", ylim=(0, 8000), title="STFT log magnitude (dB)")
image = axes[2].pcolormesh(time, np.arange(64), mel_db, shading="auto", cmap="magma", vmin=-85, vmax=0)
axes[2].set(xlabel="Time (s)", ylabel="Mel band", title="64-band log-mel image (dB)")
fig.colorbar(image, ax=axes[2], label="dB")
plt.show()


## 3. Paint on the real image

We paint the STFT image because its frequency bins can be paired with phase for playback; mel bands compress those bins and are harder to invert. The original STFT image is copied to `edited_db`. Each brush adds or subtracts decibels in a soft elliptical patch. Edit the examples or add your own calls; the texture of the actual recording remains visible. Try a narrow horizontal brush to alter a tone and a broad vertical brush to alter a transient. To start from a nearly blank image instead, uncomment the marked line.


In [ ]:
edited_db = stft_db.copy()
edited_db[:] = -85  # optional: paint a new image from a quiet canvas

def brush(center_s, center_hz, width_s, width_hz, change_db):
    """Add a smooth bright or dark patch to the editable sound image."""
    patch = np.exp(-0.5 * (
        ((time[None, :] - center_s) / width_s)**2 +
        ((frequency[:, None] - center_hz) / width_hz)**2
    ))
    edited_db[:] += change_db * patch

brush(center_s=1.2, center_hz=1800, width_s=0.25, width_hz=450, change_db=-24)
brush(center_s=3.3, center_hz=6100, width_s=0.45, width_hz=300, change_db=30)
edited_db = np.clip(edited_db, -100, 10)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True, sharey=True, constrained_layout=True)
for ax, image, title in zip(axes, [stft_db, edited_db], ["Original recording", "Your edited image"]):
    ax.pcolormesh(time, frequency, image, shading="auto", cmap="magma", vmin=-85, vmax=0)
    ax.set(xlabel="Time (s)", ylabel="Frequency (Hz)", title=title, ylim=(0, 8000))
plt.show()


## 4. Hear the edited image

We reuse the recording's original phase to make a stable preview. That works well for editing an existing image. A completely new magnitude image has no phase of its own and needs phase estimation (for example, Griffin–Lim); many different waveforms can share nearly the same magnitude image.


In [ ]:
edited_magnitude = 10 ** (edited_db / 20)
edited_spectrum = edited_magnitude * np.exp(1j * np.angle(spectrum))
_, edited_audio = signal.istft(
    edited_spectrum, fs=sr, nperseg=n_fft, noverlap=n_fft-hop,
    
    input_onesided=True, boundary=True
)
edited_audio = np.pad(edited_audio[:len(x)], (0, max(0, len(x)-len(edited_audio))))
assert len(edited_audio) == len(x) and np.isfinite(edited_audio).all()
print("Original")
display(Audio(x, rate=sr))
print("Edited")
display(Audio(edited_audio, rate=sr))


### Try this

1. Darken a short vertical patch across many frequencies. Which audible event changes?
2. Brighten a horizontal band, then move it upward. How does the sound change?
3. Set `edited_db[:] = -85`, draw several brushes, and listen. Why is the result less natural than the original?
4. Compare the STFT image with the log-mel image. Which fine details disappear when frequency bins are pooled into mel bands?
